# 13 — Alternative Constraint Handling

**Maps to:** `report/Chapters/Task2.tex` §`T2:Alternative`.  
**Ticket:** TICKET-13.

Implement at least one of: penalty function, feasibility-preserving operator. Pick penalty if behind schedule (fastest to implement).

Used by comparative experiment (TICKET-17).

In [1]:
import numpy as np
from pathlib import Path
from dataclasses import dataclass
import tsplib95

In [2]:
def load_tsp(path):
    problem     = tsplib95.load(str(path))
    nodes       = list(problem.get_nodes())
    coords      = np.array(
        [problem.node_coords[node] for node in nodes],
        dtype=np.float64
    )
    diff        = coords[:, np.newaxis, :] - coords[np.newaxis, :, :]
    dist_matrix = np.sqrt((diff**2).sum(axis=-1))
    return coords, dist_matrix


def tour_length(tour, dist_matrix):
    return dist_matrix[tour, np.roll(tour, -1)].sum()


def random_population(pop_size, n_cities, rng):
    return np.array([rng.permutation(n_cities) for _ in range(pop_size)])


def tournament_select(population, fitnesses, k, rng):
    candidates = rng.choice(len(population), size=k, replace=False)
    winner     = candidates[np.argmin(fitnesses[candidates])]
    return population[winner].copy()


def pmx(parent_a, parent_b, pt1, pt2):
    n     = len(parent_a)
    child = np.full(n, -1, dtype=parent_a.dtype)
    child[pt1:pt2 + 1] = parent_a[pt1:pt2 + 1]
    segment_vals = set(child[pt1:pt2 + 1])
    for i in range(pt1, pt2 + 1):
        val = parent_b[i]
        if val not in segment_vals:
            j = i
            while pt1 <= j <= pt2:
                j = np.where(parent_b == parent_a[j])[0][0]
            child[j] = val
    empty        = child == -1
    child[empty] = parent_b[empty]
    return child


def swap_mutation(tour, rate, rng):
    tour = tour.copy()
    if rng.random() < rate:
        i, j        = rng.choice(len(tour), size=2, replace=False)
        tour[i], tour[j] = tour[j], tour[i]
    return tour

**Implementation**

The penalty function approach is selected as the alternative constraint-handling strategy for this study.

Unlike the repair mechanism, which corrects infeasible chromosomes before fitness evaluation, the penalty function allows infeasible solutions to remain in the population but assigns them a higher fitness value as a form of punishment. This discourages the GA from selecting infeasible solutions without completely eliminating them from the search process.

`penalty(chromosome, dist_matrix, weight)` computes the penalised fitness of a chromosome. It first calculates the base tour length, then counts the number of constraint violations, specifically duplicate cities in the tour. The final fitness is the sum of the base tour length and a penalty term proportional to the number of violations multiplied by the penalty weight.

In [3]:
def penalty(chromosome, dist_matrix, weight):
    n_cities   = dist_matrix.shape[0]
    base       = tour_length(chromosome, dist_matrix)
    violations = n_cities - len(set(chromosome))
    return base + weight * violations


def penalised_fitness(population, dist_matrix, weight):
    return np.array([penalty(ind, dist_matrix, weight) for ind in population])

**Unit Test**

The penalty function is verified using two chromosomes, one valid and one infeasible, to confirm that violations are correctly detected and penalised.

In [4]:
rng          = np.random.default_rng(42)
dist_4       = np.array([
    [0, 1, 2, 3],
    [1, 0, 1, 2],
    [2, 1, 0, 1],
    [3, 2, 1, 0],
], dtype=np.float64)

valid_tour    = np.array([0, 1, 2, 3])
invalid_tour  = np.array([0, 1, 1, 3])  # bandar 1 ulang, bandar 2 missing

weight = 1000.0

valid_fitness   = penalty(valid_tour,   dist_4, weight)
invalid_fitness = penalty(invalid_tour, dist_4, weight)

print(f"Valid tour    : {valid_tour}   fitness = {valid_fitness:.2f}")
print(f"Invalid tour  : {invalid_tour}  fitness = {invalid_fitness:.2f}")

try:
    assert valid_fitness < invalid_fitness, \
        "Valid tour should have lower fitness than invalid tour"
    assert penalty(valid_tour, dist_4, weight) == tour_length(valid_tour, dist_4), \
        "Valid tour should have no penalty"
    print("penalty test passed.")
except AssertionError as e:
    print(f"penalty test failed: {e}")

Valid tour    : [0 1 2 3]   fitness = 6.00
Invalid tour  : [0 1 1 3]  fitness = 1006.00
penalty test passed.


**Demo on kroA100**

To demonstrate the effect of the penalty approach, a GA run is performed on kroA100 using penalised fitness instead of repair. The result is compared against the repair-based GA to highlight the difference in solution quality and convergence behaviour.

In [5]:
OPTIMAL = 21282  # kroA100 known optimal (TSPLIB)

@dataclass
class GAConfig:
    pop_size      : int   = 50
    n_generations : int   = 100
    crossover_rate: float = 0.8
    mutation_rate : float = 0.1
    tournament_k  : int   = 3
    elitism_count : int   = 1
    penalty_weight: float = 50000.0
    seed          : int   = 42


def run_ga_penalty(config, dist_matrix):
    n_cities   = dist_matrix.shape[0]
    rng        = np.random.default_rng(config.seed)
    population = random_population(config.pop_size, n_cities, rng)
    fitnesses  = penalised_fitness(population, dist_matrix, config.penalty_weight)
    history    = {"best": [], "mean": []}

    for gen in range(config.n_generations):
        history["best"].append(float(fitnesses.min()))
        history["mean"].append(float(fitnesses.mean()))

        new_pop    = []
        elite_idx  = np.argsort(fitnesses)[:config.elitism_count]
        for idx in elite_idx:
            new_pop.append(population[idx].copy())

        while len(new_pop) < config.pop_size:
            p1  = tournament_select(population, fitnesses, config.tournament_k, rng)
            p2  = tournament_select(population, fitnesses, config.tournament_k, rng)
            pts = sorted(rng.choice(n_cities, size=2, replace=False))

            if rng.random() < config.crossover_rate:
                child = pmx(p1, p2, pts[0], pts[1])
            else:
                child = p1.copy()

            child = swap_mutation(child, config.mutation_rate, rng)
            new_pop.append(child)

        population = np.array(new_pop[:config.pop_size])
        fitnesses  = penalised_fitness(population, dist_matrix, config.penalty_weight)

    history["best"].append(float(fitnesses.min()))
    history["mean"].append(float(fitnesses.mean()))

    best_idx = np.argmin(fitnesses)
    return {
        "best_tour"   : population[best_idx],
        "best_fitness": float(tour_length(population[best_idx], dist_matrix)),
        "history"     : history,
    }


coords, dist_matrix = load_tsp(Path('../data/TSP-dataset/kroA100.tsp'))
config              = GAConfig()
result              = run_ga_penalty(config, dist_matrix)

print(f"Best tour length : {result['best_fitness']:.2f}")
print(f"Known optimal    : {OPTIMAL:,}")
print(f"Gap              : {(result['best_fitness'] - OPTIMAL) / OPTIMAL * 100:.1f}%")

Best tour length : 86346.04
Known optimal    : 21,282
Gap              : 305.7%


**END**